# Hugging Face Brainrot / Gen-Z Dataset Normalization

This notebook loads selected Hugging Face datasets, selects the essential fields for the slang pipeline, normalizes every dataset into the same column types, merges them, deduplicates them, and saves database-ready outputs.

## Essential unified fields

- `text`: primary slang / brainrot / Gen-Z language text used for analysis.
- `standard_text`: standard English source text only when the original dataset provides a real paired source sentence.
- `context`: optional title, conversation metadata, or source context.
- `role`: message role for conversation datasets, such as `user` or `assistant`.
- `category`: dataset-provided category or derived type.
- `source_dataset`: Hugging Face dataset ID.
- `source_url`: Hugging Face dataset URL.
- `source_file`: file loaded from the dataset repository.
- `split`: train / validation / test / full when available.
- `collected_at`: notebook processing date.

## Decision on `standard_text`

Do not automatically fill missing `standard_text`. Conversation and corpus datasets do not provide a reliable plain-English equivalent, and generating one would create synthetic data that can bias linguistic analysis. Keep `standard_text` as an optional empty field in the full merged dataset, and use the separate `parallel_df` export when a task requires paired `standard_text` / `text` rows.

In [1]:
from __future__ import annotations
%pip install pandas huggingface-hub fsspec


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import ast
import re
from datetime import date
from pathlib import Path
from typing import Any

import pandas as pd

In [3]:
COLLECTED_AT = date.today().isoformat()
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_COLUMNS = [
    "text",
    "standard_text",
    "context",
    "role",
    "category",
    "source_dataset",
    "source_url",
    "source_file",
    "split",
    "collected_at",
]

HF_SOURCES = [
    {
        "name": "grenishrai/brainrot-conversation",
        "kind": "messages",
        "path": "hf://datasets/grenishrai/brainrot-conversation/data_min.jsonl",
        "source_file": "data_min.jsonl",
        "split": "full",
    },
    {
        "name": "ethan00alphayehah/brainrot-dataset",
        "kind": "source_target",
        "path": "hf://datasets/ethan00alphayehah/brainrot-dataset/train.jsonl",
        "source_file": "train.jsonl",
        "split": "train",
        "source_column": "source",
        "target_column": "target",
    },
    {
        "name": "Tralalabs/brainrot-smoll-corpus-jsonl",
        "kind": "corpus",
        "path": "hf://datasets/Tralalabs/brainrot-smoll-corpus-jsonl/mayo_2026_corpus.jsonl",
        "source_file": "mayo_2026_corpus.jsonl",
        "split": "full",
        "text_column": "text",
        "context_column": "title",
        "category_column": "category",
        "category_filter": "characters",
    },
    {
        "name": "Andy-ML-And-AI/gen-alpha-brainrot",
        "kind": "messages",
        "path": "hf://datasets/Andy-ML-And-AI/gen-alpha-brainrot/dataset.jsonl",
        "source_file": "dataset.jsonl",
        "split": "full",
    },
    {
        "name": "projolx/genz_brainrot_dataset",
        "kind": "source_target",
        "path": "hf://datasets/projolx/genz_brainrot_dataset/genz_dataset_cleaned_1.csv",
        "source_file": "genz_dataset_cleaned_1.csv",
        "split": "full",
        "source_column": "normal",
        "target_column": "gen_z",
    },
    {
        "name": "shvn22k/brainrot-dataset",
        "kind": "source_target",
        "path": "hf://datasets/shvn22k/brainrot-dataset/train.jsonl",
        "source_file": "train.jsonl",
        "split": "train",
        "source_column": "source",
        "target_column": "target",
    },
]

In [4]:
def clean_text(value: Any) -> str:
    """Return normalized text while preserving slang spelling and punctuation."""
    if value is None or pd.isna(value):
        return ""
    text = str(value)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def hf_url(dataset_name: str) -> str:
    return f"https://huggingface.co/datasets/{dataset_name}"


def read_hf_dataframe(config: dict[str, Any]) -> pd.DataFrame:
    """Load one Hugging Face file using pandas and the hf:// filesystem."""
    path = config["path"]
    if path.endswith(".csv"):
        return pd.read_csv(path)
    return pd.read_json(path, lines=True)


def parse_messages(value: Any) -> list[dict[str, str]]:
    """Normalize chat-style message arrays from JSONL datasets."""
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (SyntaxError, ValueError):
            return []
    if not isinstance(value, list):
        return []

    messages = []
    for item in value:
        if not isinstance(item, dict):
            continue
        role = clean_text(item.get("role"))
        content = clean_text(item.get("content"))
        if content:
            messages.append({"role": role, "content": content})
    return messages


def enforce_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Guarantee the merged dataframe has consistent columns and string dtypes."""
    for column in UNIFIED_COLUMNS:
        if column not in df.columns:
            df[column] = ""
    df = df[UNIFIED_COLUMNS].copy()
    for column in UNIFIED_COLUMNS:
        df[column] = df[column].fillna("").astype("string")
    return df

In [5]:
def normalize_messages_dataset(df: pd.DataFrame, config: dict[str, Any]) -> pd.DataFrame:
    """Explode conversation datasets into one row per useful message."""
    rows = []
    for original_index, row in df.iterrows():
        for turn_index, message in enumerate(parse_messages(row.get("messages"))):
            role = message["role"]
            text = message["content"]

            # System prompts describe assistant behavior, not language samples.
            if role.casefold() == "system":
                continue

            rows.append({
                "text": text,
                "standard_text": "",
                "context": f"conversation_row={original_index}; turn={turn_index}",
                "role": role,
                "category": "conversation_message",
                "source_dataset": config["name"],
                "source_url": hf_url(config["name"]),
                "source_file": config["source_file"],
                "split": config["split"],
                "collected_at": COLLECTED_AT,
            })

    return enforce_schema(pd.DataFrame(rows))


def normalize_source_target_dataset(df: pd.DataFrame, config: dict[str, Any]) -> pd.DataFrame:
    """Normalize paired standard-English to Gen-Z/brainrot translation datasets."""
    source_column = config["source_column"]
    target_column = config["target_column"]

    rows = []
    for original_index, row in df.iterrows():
        text = clean_text(row.get(target_column))
        standard_text = clean_text(row.get(source_column))
        if not text:
            continue
        rows.append({
            "text": text,
            "standard_text": standard_text,
            "context": f"paired_row={original_index}",
            "role": "",
            "category": "parallel_translation",
            "source_dataset": config["name"],
            "source_url": hf_url(config["name"]),
            "source_file": config["source_file"],
            "split": config["split"],
            "collected_at": COLLECTED_AT,
        })

    return enforce_schema(pd.DataFrame(rows))


def normalize_corpus_dataset(df: pd.DataFrame, config: dict[str, Any]) -> pd.DataFrame:
    """Normalize corpus/reference datasets with text, title, and category fields."""
    category_column = config.get("category_column")
    category_filter = config.get("category_filter")

    if category_column and category_filter and category_column in df.columns:
        df = df[df[category_column].astype("string").str.casefold() == category_filter.casefold()].copy()

    rows = []
    for original_index, row in df.iterrows():
        text = clean_text(row.get(config["text_column"]))
        if not text:
            continue
        rows.append({
            "text": text,
            "standard_text": "",
            "context": clean_text(row.get(config.get("context_column", ""))),
            "role": "",
            "category": clean_text(row.get(category_column)) if category_column else "reference_corpus",
            "source_dataset": config["name"],
            "source_url": hf_url(config["name"]),
            "source_file": config["source_file"],
            "split": config["split"],
            "collected_at": COLLECTED_AT,
        })

    return enforce_schema(pd.DataFrame(rows))

In [6]:
def normalize_dataset(df: pd.DataFrame, config: dict[str, Any]) -> pd.DataFrame:
    if config["kind"] == "messages":
        return normalize_messages_dataset(df, config)
    if config["kind"] == "source_target":
        return normalize_source_target_dataset(df, config)
    if config["kind"] == "corpus":
        return normalize_corpus_dataset(df, config)
    raise ValueError(f"Unsupported dataset kind: {config['kind']}")


raw_frames: dict[str, pd.DataFrame] = {}
normalized_frames: list[pd.DataFrame] = []
load_errors: list[dict[str, str]] = []

for config in HF_SOURCES:
    try:
        raw_df = read_hf_dataframe(config)
        normalized_df = normalize_dataset(raw_df, config)
        raw_frames[config["name"]] = raw_df
        normalized_frames.append(normalized_df)
        print(f"Loaded {config['name']}: raw={raw_df.shape}, normalized={normalized_df.shape}")
    except Exception as exc:
        load_errors.append({"dataset": config["name"], "error": str(exc)})
        print(f"Skipped {config['name']}: {exc}")

if not normalized_frames:
    raise RuntimeError("No Hugging Face datasets were loaded. Check network access and dataset paths.")

merged_df = pd.concat(normalized_frames, ignore_index=True)
merged_df = enforce_schema(merged_df)

before_dedupe = len(merged_df)
dedupe_columns = ["text", "standard_text", "role", "category", "source_dataset"]
merged_df = merged_df.drop_duplicates(subset=dedupe_columns).reset_index(drop=True)
parallel_df = merged_df[merged_df["standard_text"].str.len() > 0].copy().reset_index(drop=True)

print(f"Merged rows before dedupe: {before_dedupe}")
print(f"Merged rows after dedupe: {len(merged_df)}")
print(f"Rows with real standard_text: {len(parallel_df)}")

if load_errors:
    print("Datasets skipped during load:")
    for item in load_errors:
        print(f"- {item['dataset']}: {item['error']}")

Loaded grenishrai/brainrot-conversation: raw=(2005, 1), normalized=(6026, 10)
Loaded ethan00alphayehah/brainrot-dataset: raw=(5663, 2), normalized=(5663, 10)
Loaded Tralalabs/brainrot-smoll-corpus-jsonl: raw=(41, 5), normalized=(7, 10)
Loaded Andy-ML-And-AI/gen-alpha-brainrot: raw=(600, 1), normalized=(1200, 10)
Loaded projolx/genz_brainrot_dataset: raw=(3170, 2), normalized=(3170, 10)
Loaded shvn22k/brainrot-dataset: raw=(5663, 2), normalized=(5663, 10)
Merged rows before dedupe: 21729
Merged rows after dedupe: 20931
Rows with real standard_text: 14492


In [8]:
# Quick quality checks for the unified schema.
print(merged_df.dtypes)
print("Rows by source dataset:")
display(merged_df.groupby("source_dataset").size().reset_index(name="rows"))

print("Rows by category:")
display(merged_df.groupby("category").size().reset_index(name="rows"))

print("Sample merged records:")
display(merged_df.head(10))

print("Sample parallel records with standard_text:")
display(parallel_df.head(10))

text              string[python]
standard_text     string[python]
context           string[python]
role              string[python]
category          string[python]
source_dataset    string[python]
source_url        string[python]
source_file       string[python]
split             string[python]
collected_at      string[python]
dtype: object
Rows by source dataset:


,source_dataset,rows
0,Andy-ML-And-AI/gen-alpha-brainrot,1099
1,Tralalabs/brainrot-smoll-corpus-jsonl,7
2,ethan00alphayehah/brainrot-dataset,5661
3,grenishrai/brainrot-conversation,5333
4,projolx/genz_brainrot_dataset,3170
5,shvn22k/brainrot-dataset,5661


Rows by category:


,category,rows
0,characters,7
1,conversation_message,6432
2,parallel_translation,14492


Sample merged records:


,text,standard_text,context,role,category,source_dataset,source_url,source_file,split,collected_at
0,bro i spilled coffee on my white shirt,,conversation_row=0; turn=1,user,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
1,"lmao that's straight up skibidi rizz fail, hit...",,conversation_row=0; turn=2,assistant,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
2,it's ruined now,,conversation_row=0; turn=3,user,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
3,"nah fanum tax that stain, turn it into a gyatt...",,conversation_row=0; turn=4,assistant,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
4,my crush ghosted me,,conversation_row=1; turn=1,user,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
5,"brutal they pulling grimace shake vibes, you'r...",,conversation_row=1; turn=2,assistant,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
6,what do i do,,conversation_row=1; turn=3,user,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
7,"level up the grindset, next one's got that mog...",,conversation_row=1; turn=4,assistant,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
8,i hate mondays,,conversation_row=2; turn=1,user,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11
9,mondays be giving skibidi toilet energy switch...,,conversation_row=2; turn=2,assistant,conversation_message,grenishrai/brainrot-conversation,https://huggingface.co/datasets/grenishrai/bra...,data_min.jsonl,full,2026-05-11


Sample parallel records with standard_text:


,text,standard_text,context,role,category,source_dataset,source_url,source_file,split,collected_at
0,bro i’m running on vibes and resentment rn,I am very tired today,paired_row=0,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
1,"academia is a scam, i’m gonna sell stickers",I don't want to study anymore,paired_row=1,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
2,bruh this assignment got more twists than dark,This is extremely difficult,paired_row=2,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
3,me when life life’d too hard today,I am feeling sad,paired_row=3,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
4,dopamine secured 🔥 mood looking like stock mar...,I am very happy today,paired_row=4,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
5,did i install the paid version of suffering or...,Why is this happening to me?,paired_row=5,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
6,imma go live in the mountains w a goat,I can't take this anymore,paired_row=6,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
7,gng grind never stops even when i do,I am trying my best,paired_row=7,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
8,alt f4 on reality tbh,I give up,paired_row=8,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11
9,insomnia unlocked new DLC: existential thought...,Please let me sleep,paired_row=9,,parallel_translation,ethan00alphayehah/brainrot-dataset,https://huggingface.co/datasets/ethan00alphaye...,train.jsonl,train,2026-05-11


In [9]:
CSV_OUTPUT = OUTPUT_DIR / "huggingface_slang_dataset.csv"
JSON_OUTPUT = OUTPUT_DIR / "huggingface_slang_dataset.json"
PARALLEL_CSV_OUTPUT = OUTPUT_DIR / "huggingface_parallel_dataset.csv"
PARALLEL_JSON_OUTPUT = OUTPUT_DIR / "huggingface_parallel_dataset.json"

merged_df.to_csv(CSV_OUTPUT, index=False, encoding="utf-8", na_rep="")
merged_df.to_json(JSON_OUTPUT, orient="records", indent=2, force_ascii=False)
parallel_df.to_csv(PARALLEL_CSV_OUTPUT, index=False, encoding="utf-8", na_rep="")
parallel_df.to_json(PARALLEL_JSON_OUTPUT, orient="records", indent=2, force_ascii=False)

print(f"Saved full CSV to {CSV_OUTPUT}")
print(f"Saved full JSON to {JSON_OUTPUT}")
print(f"Saved parallel-only CSV to {PARALLEL_CSV_OUTPUT}")
print(f"Saved parallel-only JSON to {PARALLEL_JSON_OUTPUT}")

Saved full CSV to data\processed\huggingface_slang_dataset.csv
Saved full JSON to data\processed\huggingface_slang_dataset.json
Saved parallel-only CSV to data\processed\huggingface_parallel_dataset.csv
Saved parallel-only JSON to data\processed\huggingface_parallel_dataset.json
